In [ ]:
import requests
import pandas as pd

In [ ]:
# Fetch fixtures data
FIXTURES_URL = 'https://fantasy.premierleague.com/api/fixtures/'
response = requests.get(FIXTURES_URL)
fixtures = response.json()

In [8]:
# List to hold all teams' weekly data
all_teams_weekly_data = []

# Loop through each fixture and collect statistics for each team
for fixture in fixtures:
    gameweek = fixture['event']
    home_team_id = fixture['team_h']
    away_team_id = fixture['team_a']
    home_team_stats = {
        'gameweek': gameweek,
        'team_id': home_team_id,
        'team_type': 'home',
        'total_goals': fixture['team_h_score'],
        'opponent_goals': fixture['team_a_score'],
        'clean_sheet': 1 if fixture['team_a_score'] == 0 else 0,
        'xG': fixture.get('team_h_expected_goals', None),  # Expected goals (if available)
        'xGC': fixture.get('team_h_expected_goals_conceded', None),  # Expected goals conceded (if available)
        'FDR': fixture['team_h_difficulty'],  # Placeholder for Fixture Difficulty Rating
    }
    away_team_stats = {
        'gameweek': gameweek,
        'team_id': away_team_id,
        'team_type': 'away',
        'total_goals': fixture['team_a_score'],
        'opponent_goals': fixture['team_h_score'],
        'clean_sheet': 1 if fixture['team_h_score'] == 0 else 0,
        'xG': fixture.get('team_a_expected_goals', None),  # Expected goals (if available)
        'xGC': fixture.get('team_a_expected_goals_conceded', None),  # Expected goals conceded (if available)
        'FDR': fixture['team_a_difficulty'],  # Placeholder for Fixture Difficulty Rating
    }
    all_teams_weekly_data.append(home_team_stats)
    all_teams_weekly_data.append(away_team_stats)

# Create DataFrame from the weekly data
weekly_teams_df = pd.DataFrame(all_teams_weekly_data)

# Extract team information for mapping team names
FPL_API_URL = 'https://fantasy.premierleague.com/api/bootstrap-static/'
response = requests.get(FPL_API_URL)
data = response.json()
teams = data['teams']
teams_df = pd.DataFrame(teams)
teams_df = teams_df[['id', 'name']]

# Merge team names into the weekly data DataFrame
weekly_teams_df = weekly_teams_df.merge(teams_df, left_on='team_id', right_on='id')
weekly_teams_df.drop(columns=['id'], inplace=True)

# Select and rename relevant columns
weekly_teams_df = weekly_teams_df[['gameweek', 'team_id', 'name', 'team_type', 'total_goals', 'opponent_goals', 'clean_sheet', 'xG', 'xGC', 'FDR']]
weekly_teams_df.columns = ['Gameweek', 'Team ID', 'Team Name', 'Team Type', 'Total Goals', 'Opponent Goals', 'Clean Sheet', 'xG', 'xGC', 'FDR']

# Save to a CSV file
weekly_teams_df.to_csv('fpl_team_weekly_data.csv', index=False)

# Display a sample of the DataFrame
print(weekly_teams_df.head())


   Gameweek  Team ID  Team Name Team Type  Total Goals  Opponent Goals  \
0         1       14    Man Utd      home            1               0   
1         1        9     Fulham      away            0               1   
2         1       10    Ipswich      home            0               2   
3         1       12  Liverpool      away            2               0   
4         1        1    Arsenal      home            2               0   

   Clean Sheet    xG   xGC  FDR  
0            1  None  None    3  
1            0  None  None    3  
2            0  None  None    5  
3            1  None  None    2  
4            1  None  None    3  


In [10]:
print(weekly_teams_df.tail())


     Gameweek  Team ID  Team Name Team Type  Total Goals  Opponent Goals  \
755        38        1    Arsenal      away            2               1   
756        38       18      Spurs      home            1               4   
757        38        5   Brighton      away            4               1   
758        38       20     Wolves      home            1               1   
759        38        4  Brentford      away            1               1   

     Clean Sheet    xG   xGC  FDR  
755            0  None  None    1  
756            0  None  None    3  
757            0  None  None    3  
758            0  None  None    3  
759            0  None  None    3  


In [12]:
fixture

{'code': 2444849,
 'event': 38,
 'finished': True,
 'finished_provisional': True,
 'id': 380,
 'kickoff_time': '2025-05-25T15:00:00Z',
 'minutes': 90,
 'provisional_start_time': False,
 'started': True,
 'team_a': 4,
 'team_a_score': 1,
 'team_h': 20,
 'team_h_score': 1,
 'stats': [{'identifier': 'goals_scored',
   'a': [{'value': 1, 'element': 99}],
   'h': [{'value': 1, 'element': 770}]},
  {'identifier': 'assists',
   'a': [{'value': 1, 'element': 101}],
   'h': [{'value': 1, 'element': 566}]},
  {'identifier': 'own_goals', 'a': [], 'h': []},
  {'identifier': 'penalties_saved', 'a': [], 'h': []},
  {'identifier': 'penalties_missed', 'a': [], 'h': []},
  {'identifier': 'yellow_cards',
   'a': [{'value': 1, 'element': 110}],
   'h': [{'value': 1, 'element': 553}, {'value': 1, 'element': 559}]},
  {'identifier': 'red_cards', 'a': [], 'h': []},
  {'identifier': 'saves',
   'a': [{'value': 5, 'element': 91}],
   'h': [{'value': 6, 'element': 554}]},
  {'identifier': 'bonus',
   'a': [{'v